In [ ]:
!pip install transformers
!pip install statsmodels


In [ ]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from transformers import DistilBertModel
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

### Note: Executed in Google Colab

This notebook was run in Google Colab rather than locally, due to environment compatibility issues encountered when trying to run DistilBERT (PyTorch) locally — a metadata-detection mismatch between `transformers` and the local `torch` installation that surfaced as `is_torch_available()` returning `False` despite `torch` being correctly installed and importable. Combined with confirming the local machine has no GPU (`torch` reported as the CPU-only build), Colab's free GPU and pre-configured, version-matched PyTorch environment made it the more practical choice for training and evaluating this model.

In [ ]:
print("GPU available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

GPU available: True
Device name: Tesla T4


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:

try:
    train_df = pd.read_csv('data/train_clean.csv')
    test_df = pd.read_csv('data/test_clean.csv')
    print("Loaded from local 'data/' directory (VS Code path)")
except FileNotFoundError:
    train_df = pd.read_csv('train_clean.csv')
    test_df = pd.read_csv('test_clean.csv')
    print("Loaded from root directory (Colab upload path)")

print(train_df.shape)
print(test_df.shape)

Loaded from root directory (Colab upload path)
(161297, 7)
(53766, 7)


### Why a Transformer-Based Model (BERT)

A comparative study on medical-notes text classification under class imbalance ([Lu, Ehwerhemuepha & Rakovski, 2022](https://pmc.ncbi.nlm.nih.gov/articles/PMC9250736/)) evaluated seven architectures — CNN, RNN, GRU, LSTM, Bi-LSTM, a Transformer encoder, and BERT-Base — and found the attention-based encoder family (Transformer encoder and BERT) outperformed the RNN/CNN family in nearly all of 16 imbalanced classification tasks. Given our own EDA confirmed a similarly imbalanced rating distribution, this motivated choosing an encoder-based model for our task.

Within the encoder family, we chose **BERT (pretrained)** over a Transformer encoder trained from scratch. Transformers lack the built-in structural assumptions RNNs and CNNs have, so without pretraining they typically need very large datasets to learn well. A pretrained model, by contrast, already carries a statistical understanding of language learned from a much larger corpus than we could provide, making fine-tuning more sample-efficient and generally stronger than training an encoder from scratch unless working with substantially more data than we have.

Notably, BERT-Base itself performed poorly in that study — due to being pretrained on general text rather than clinical language, and the notes' average length (557 words) exceeding BERT's 512-token limit. Neither issue applies to us: our reviews are everyday patient language, not jargon, and our 95th percentile length (146 words) is well under BERT's limit. So the domain-mismatch that hurt BERT there is unlikely to carry over here.

### Why DistilBERT (Lightweight Version)

This decision was based on the original DistilBERT paper ([Sanh, Debut, Chaumond & Wolf, 2019](https://arxiv.org/abs/1910.01108)), which introduced DistilBERT as a distilled version of BERT trained via knowledge distillation from a full BERT teacher model. The paper reports DistilBERT retains approximately 97% of BERT's language understanding performance on the GLUE benchmark while being 40% smaller (66M vs 110M parameters) and 60% faster at inference.

This tradeoff is also confirmed on a task directly comparable to ours: a study on Yelp review star-rating prediction ([arXiv:2012.06690](https://arxiv.org/abs/2012.06690)) — free-text review to numeric rating, the same task structure as our assignment — found DistilBERT scored only about 0.5% lower in accuracy than full BERT, while training at nearly double the speed.

Given our project's compute constraints (no local GPU, free-tier Colab GPU access), and that the accuracy cost of DistilBERT over full BERT is minor and well-documented on a near-identical task, we chose DistilBERT over full BERT-Base.

In [ ]:
# Stratified split keeps the same imbalanced rating distribution in both train and validation

train_split, val_split = train_test_split(
    train_df, test_size=0.15, stratify=train_df['rating'], random_state=42
)
print(f"Train: {len(train_split)}, Val: {len(val_split)}")
print(val_split['rating'].value_counts(normalize=True).sort_index())

Train: 137102, Val: 24195
rating
1     0.134036
2     0.042984
3     0.040380
4     0.031081
5     0.049680
6     0.039306
7     0.058607
8     0.117132
9     0.170696
10    0.316098
Name: proportion, dtype: float64


In [ ]:
# Class weights computed from train split only, to avoid leaking val-set info into training decisions
labels_0indexed = train_split['rating'].values - 1  # ratings are 1-10, model expects 0-9

class_weights = compute_class_weight(
    class_weight='balanced', classes=np.arange(10), y=labels_0indexed
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(class_weights)

[0.74609273 2.32731285 2.47655347 3.21835681 2.01294964 2.54269288
 1.70567305 0.85389885 0.58588095 0.31633326]


Note: the EDA-derived vocabulary size (`min_frequency=5`, 18,581 words) applied to the from-scratch embedding approach (GRU/BiLSTM), not here. DistilBERT uses its own fixed, pretrained WordPiece vocabulary (30,522 tokens), which cannot be resized without discarding the pretrained weights this model's value depends on.

Note: the EDA's punctuation-stripping tokenization (used to estimate the from-scratch embedding vocabulary) also does not apply here. DistilBERT's WordPiece tokenizer already handles punctuation natively (splitting it into separate tokens rather than merging or discarding it), and was pretrained on naturally-punctuated text. Manually stripping punctuation before tokenizing would create a mismatch with that pretraining and risks losing genuine signal (e.g., emphasis conveyed through punctuation), so we pass review text to the tokenizer unmodified beyond the HTML-entity cleanup already done in `01_data_cleaning.ipynb`.

In [ ]:
#was injected here in the re run so we wont need to train the BERT model again
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 160
BATCH_SIZE = 32

# Load directly from the saved best checkpoint — skips training entirely
model = DistilBertForSequenceClassification.from_pretrained('distilbert_10class_best').to(device)
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert_10class_best')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
def tokenize_split(df):
    texts = df['review'].tolist()
    labels = torch.tensor(df['rating'].values - 1)
    # max_length=160 truncates here only — train_split/val_split['review'] stays untouched
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors='pt')
    return encodings, labels

train_encodings, train_labels = tokenize_split(train_split)
val_encodings, val_labels = tokenize_split(val_split)

train_dataset = TensorDataset(train_encodings['input_ids'], train_encodings['attention_mask'], train_labels)
val_dataset = TensorDataset(val_encodings['input_ids'], val_encodings['attention_mask'], val_labels)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)  # no shuffle needed for validation

In [ ]:
import torch.nn as nn

model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=10).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
scaler = torch.amp.GradScaler('cuda')
# Computing loss manually (not via model's built-in labels= argument) since that path doesn't support class weights
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
EPOCHS = 4

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for input_ids, attention_mask, batch_labels in train_loader:
        input_ids, attention_mask, batch_labels = input_ids.to(device), attention_mask.to(device), batch_labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, batch_labels)  # weighted loss applied here
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} - avg train loss: {total_loss/len(train_loader):.4f}")

Epoch 1/4 - avg train loss: 1.8618
Epoch 2/4 - avg train loss: 1.6439
Epoch 3/4 - avg train loss: 1.4523
Epoch 4/4 - avg train loss: 1.2605


In [ ]:
model.save_pretrained('distilbert_10class_v1')
tokenizer.save_pretrained('distilbert_10class_v1')

# Also save optimizer + scaler state if you want to resume training with identical dynamics later
torch.save({
    'optimizer_state_dict': optimizer.state_dict(),
    'scaler_state_dict': scaler.state_dict(),
    'epoch': EPOCHS,
}, 'distilbert_10class_v1/training_state.pt')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Resume from the saved checkpoint

model = DistilBertForSequenceClassification.from_pretrained('distilbert_10class_v1').to(device)
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert_10class_v1')

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
scaler = torch.amp.GradScaler('cuda')

checkpoint = torch.load('distilbert_10class_v1/training_state.pt')
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scaler.load_state_dict(checkpoint['scaler_state_dict'])

print(f"Resumed from epoch {checkpoint['epoch']}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Resumed from epoch 4


In [ ]:
# Early stopping: train up to MAX_EPOCHS, but stop once val macro-F1 hasn't improved for PATIENCE epochs in a row
MAX_EPOCHS = 5
PATIENCE = 1  # stop after 1 epoch with no improvement, given ~12 min/epoch cost

best_macro_f1 = macro_f1  # the 0.3882 we already have at 4 epochs, as the baseline to beat
epochs_without_improvement = 0
current_epoch = checkpoint['epoch']

for i in range(MAX_EPOCHS):
    # --- train one epoch ---
    model.train()
    total_loss = 0
    for input_ids, attention_mask, batch_labels in train_loader:
        input_ids, attention_mask, batch_labels = input_ids.to(device), attention_mask.to(device), batch_labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, batch_labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    current_epoch += 1
    avg_train_loss = total_loss / len(train_loader)

    # --- evaluate on validation set ---
    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for input_ids, attention_mask, batch_labels in val_loader:
            input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(batch_labels.numpy())
    val_macro_f1 = f1_score(val_true, val_preds, average='macro')

    print(f"Epoch {current_epoch} - train loss: {avg_train_loss:.4f} - val macro-F1: {val_macro_f1:.4f}")

    # --- check for improvement ---
    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        epochs_without_improvement = 0
        model.save_pretrained('distilbert_10class_best')
        tokenizer.save_pretrained('distilbert_10class_best')
        print(f"  -> New best ({val_macro_f1:.4f}), checkpoint saved")
    else:
        epochs_without_improvement += 1
        print(f"  -> No improvement ({epochs_without_improvement}/{PATIENCE} epochs)")
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

print(f"\nFinal best macro-F1: {best_macro_f1:.4f}")

Epoch 5 - train loss: 1.0736 - val macro-F1: 0.4224


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> New best (0.4224), checkpoint saved
Epoch 6 - train loss: 0.9077 - val macro-F1: 0.4776


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> New best (0.4776), checkpoint saved
Epoch 7 - train loss: 0.7693 - val macro-F1: 0.5067


...

  -> New best (0.5067), checkpoint saved
Epoch 8 - train loss: 0.6465 - val macro-F1: 0.5223


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> New best (0.5223), checkpoint saved
Epoch 9 - train loss: 0.5590 - val macro-F1: 0.5511


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  -> New best (0.5511), checkpoint saved

Final best macro-F1: 0.5511


In [ ]:

model.eval()
all_preds, all_true = [], []
with torch.no_grad():  # no gradient tracking needed during evaluation
    for input_ids, attention_mask, batch_labels in val_loader:
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(batch_labels.numpy())

# Macro-F1 weighs all 10 classes equally, unlike accuracy which favors the majority classes — called out explicitly per our EDA decision
macro_f1 = f1_score(all_true, all_preds, average='macro')
print(f"Macro-F1: {macro_f1:.4f}\n")

print(classification_report(all_true, all_preds, target_names=[str(i) for i in range(1, 11)]))
print(confusion_matrix(all_true, all_preds))

Macro-F1: 0.5511

              precision    recall  f1-score   support

           1       0.75      0.78      0.76      3243
           2       0.50      0.58      0.54      1040
           3       0.53      0.48      0.51       977
           4       0.57      0.59      0.58       752
           5       0.46      0.56      0.51      1202
           6       0.57      0.52      0.55       951
           7       0.46      0.50      0.48      1418
           8       0.42      0.48      0.44      2834
           9       0.43      0.55      0.48      4130
          10       0.79      0.58      0.66      7648

    accuracy                           0.58     24195
   macro avg       0.55      0.56      0.55     24195
weighted avg       0.60      0.58      0.58     24195

[[2520  324  102   53   97   22   17   41   19   48]
 [ 222  606   67   34   53    6   13   21    8   10]
 [ 120  143  471   66   83   19   29   28    8   10]
 [  72   31   50  444   61   18   32   26    5   13]
 [  98   49

In [ ]:

all_preds_arr = np.array(all_preds)
all_true_arr = np.array(all_true)
within_one = np.abs(all_preds_arr - all_true_arr) <= 1
print(f"Predictions within ±1 of true rating: {within_one.mean()*100:.1f}%")

Predictions within ±1 of true rating: 83.1%


### Model B: Review + drugName + condition

Model B reuses the trained DistilBERT from Model A as a **frozen feature extractor** — its weights are not updated further. Each review is passed through it to obtain a 768-dimensional representation, which is concatenated with learned embeddings for `drugName` and `condition`, then fed into a small new classification head that is trained from scratch on top of these combined features. This avoids re-fine-tuning DistilBERT itself (the expensive part), while still testing whether adding the two categorical fields improves on Model A's text-only performance.

**Caveat specific to this model**: unlike Model A, `condition` is now actually used as an input here — which means the truncation issue documented in EDA (9.3% of rows, e.g. `"Bipolar Disorde"` vs the never-occurring `"Bipolar Disorder"`) is no longer just a cosmetic data-quality note. It now fragments what should be one category into multiple distinct embedding entries, diluting the signal `condition` could otherwise provide. We keep this caveat in mind when interpreting Model B's results.

**Architecture**: `drugName` and `condition` are each mapped through their own learned `nn.Embedding` layer — a 32-dimensional embedding for `drugName` and a 16-dimensional embedding for `condition`, both sized well below their respective vocabulary counts since neither field needs anywhere near DistilBERT's expressive capacity. These two embeddings are concatenated with the frozen 768-dimensional DistilBERT output (768 + 32 + 16 = 816 total), then passed through a small two-layer MLP head (Linear → ReLU → Dropout(0.3) → Linear) that maps down to the 10 rating classes. Only this head and the two embedding tables are trained; DistilBERT itself stays frozen throughout.

In [ ]:
# Load the trained Model A checkpoint as a frozen feature extractor — run this only after Model A finishes and is saved

text_encoder = DistilBertModel.from_pretrained('distilbert_10class_best').to(device)
tokenizer_b = DistilBertTokenizerFast.from_pretrained('distilbert_10class_best')

text_encoder.eval()
for param in text_encoder.parameters():
    param.requires_grad = False  # freeze — no gradient updates to DistilBERT itself

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert_10class_best
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.bias   | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Build categorical encodings for drugName/condition from train_split only — avoids leaking test categories into the mapping
drug_to_id = {drug: i for i, drug in enumerate(train_split['drugName'].unique())}
condition_to_id = {cond: i for i, cond in enumerate(train_split['condition'].unique())}

DRUG_UNK = len(drug_to_id)      # reserved index for any drug not seen in train
CONDITION_UNK = len(condition_to_id)  # same, for condition

def encode_drug(name):
    return drug_to_id.get(name, DRUG_UNK)

def encode_condition(name):
    return condition_to_id.get(name, CONDITION_UNK)

print(f"Unique drugs (train): {len(drug_to_id)}, Unique conditions (train): {len(condition_to_id)}")

Unique drugs (train): 3294, Unique conditions (train): 789


In [ ]:
# Extract frozen text features once per split — DistilBERT doesn't need to be re-run every epoch since it's frozen
@torch.no_grad()
def extract_text_features(encodings, batch_size=32):
    all_features = []
    for i in range(0, len(encodings['input_ids']), batch_size):
        batch_ids = encodings['input_ids'][i:i+batch_size].to(device)
        batch_mask = encodings['attention_mask'][i:i+batch_size].to(device)
        outputs = text_encoder(input_ids=batch_ids, attention_mask=batch_mask)
        pooled = outputs.last_hidden_state[:, 0, :]  # first token's hidden state, DistilBERT's [CLS]-equivalent
        all_features.append(pooled.cpu())
    return torch.cat(all_features, dim=0)

train_text_features = extract_text_features(train_encodings)  # reuses train_encodings already tokenized for Model A
val_text_features = extract_text_features(val_encodings)
print(train_text_features.shape, val_text_features.shape)

torch.Size([137102, 768]) torch.Size([24195, 768])


In [ ]:
# Build the categorical feature tensors
train_drug_ids = torch.tensor(train_split['drugName'].apply(encode_drug).values)
train_cond_ids = torch.tensor(train_split['condition'].apply(encode_condition).values)
val_drug_ids = torch.tensor(val_split['drugName'].apply(encode_drug).values)
val_cond_ids = torch.tensor(val_split['condition'].apply(encode_condition).values)

In [ ]:

class ModelB(nn.Module):
    def __init__(self, num_drugs, num_conditions, drug_dim=32, cond_dim=16, num_classes=10):
        super().__init__()
        self.drug_embedding = nn.Embedding(num_drugs + 1, drug_dim)        # +1 for the UNK index
        self.condition_embedding = nn.Embedding(num_conditions + 1, cond_dim)
        self.classifier = nn.Sequential(
            nn.Linear(768 + drug_dim + cond_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, text_features, drug_ids, condition_ids):
        drug_emb = self.drug_embedding(drug_ids)
        cond_emb = self.condition_embedding(condition_ids)
        combined = torch.cat([text_features, drug_emb, cond_emb], dim=1)
        return self.classifier(combined)

model_b = ModelB(num_drugs=len(drug_to_id), num_conditions=len(condition_to_id)).to(device)
optimizer_b = torch.optim.AdamW(model_b.parameters(), lr=1e-3)  # higher LR is fine — only training a small head, not the full transformer
loss_fn_b = nn.CrossEntropyLoss(weight=class_weights_tensor)  # same class weights as Model A

In [ ]:
MAX_EPOCHS = 50       # generous ceiling — each epoch is cheap
PATIENCE = 3           # stop after 3 epochs with no improvement

best_macro_f1_b = 0.0  # starting from scratch — no prior benchmark to beat yet
epochs_without_improvement = 0

train_dataset_b = TensorDataset(train_text_features, train_drug_ids, train_cond_ids, train_labels)
val_dataset_b = TensorDataset(val_text_features, val_drug_ids, val_cond_ids, val_labels)
train_loader_b = DataLoader(train_dataset_b, batch_size=64, shuffle=True)
val_loader_b = DataLoader(val_dataset_b, batch_size=64, shuffle=False)

for epoch in range(MAX_EPOCHS):
    model_b.train()
    total_loss = 0
    for text_feat, drug_id, cond_id, labels in train_loader_b:
        text_feat, drug_id, cond_id, labels = text_feat.to(device), drug_id.to(device), cond_id.to(device), labels.to(device)
        optimizer_b.zero_grad()
        logits = model_b(text_feat, drug_id, cond_id)
        loss = loss_fn_b(logits, labels)
        loss.backward()
        optimizer_b.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader_b)

    # validation pass —  decides when to stop
    model_b.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for text_feat, drug_id, cond_id, labels in val_loader_b:
            text_feat, drug_id, cond_id = text_feat.to(device), drug_id.to(device), cond_id.to(device)
            logits = model_b(text_feat, drug_id, cond_id)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
    val_macro_f1_b = f1_score(val_true, val_preds, average='macro')

    print(f"Epoch {epoch+1} - train loss: {avg_train_loss:.4f} - val macro-F1: {val_macro_f1_b:.4f}")

    if val_macro_f1_b > best_macro_f1_b:
        best_macro_f1_b = val_macro_f1_b
        epochs_without_improvement = 0
        torch.save(model_b.state_dict(), 'model_b_best.pt')
        print(f"  -> New best ({val_macro_f1_b:.4f}), checkpoint saved")
    else:
        epochs_without_improvement += 1
        print(f"  -> No improvement ({epochs_without_improvement}/{PATIENCE})")
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

print(f"\nFinal best Model B macro-F1: {best_macro_f1_b:.4f}")

Epoch 1 - train loss: 0.3143 - val macro-F1: 0.5657
  -> New best (0.5657), checkpoint saved
Epoch 2 - train loss: 0.2972 - val macro-F1: 0.5691
  -> New best (0.5691), checkpoint saved
Epoch 3 - train loss: 0.2909 - val macro-F1: 0.5735
  -> New best (0.5735), checkpoint saved
Epoch 4 - train loss: 0.2855 - val macro-F1: 0.5721
  -> No improvement (1/3)
Epoch 5 - train loss: 0.2801 - val macro-F1: 0.5785
  -> New best (0.5785), checkpoint saved
Epoch 6 - train loss: 0.2762 - val macro-F1: 0.5768
  -> No improvement (1/3)
Epoch 7 - train loss: 0.2704 - val macro-F1: 0.5768
  -> No improvement (2/3)
Epoch 8 - train loss: 0.2656 - val macro-F1: 0.5790
  -> New best (0.5790), checkpoint saved
Epoch 9 - train loss: 0.2601 - val macro-F1: 0.5786
  -> No improvement (1/3)
Epoch 10 - train loss: 0.2557 - val macro-F1: 0.5783
  -> No improvement (2/3)
Epoch 11 - train loss: 0.2529 - val macro-F1: 0.5777
  -> No improvement (3/3)
Early stopping triggered.

Final best Model B macro-F1: 0.5790


In [ ]:
# Same evaluation approach as Model A, for direct comparison
model_b.eval()
all_preds_b, all_true_b = [], []
with torch.no_grad():
    for text_feat, drug_id, cond_id, labels in val_loader_b:
        text_feat, drug_id, cond_id = text_feat.to(device), drug_id.to(device), cond_id.to(device)
        logits = model_b(text_feat, drug_id, cond_id)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds_b.extend(preds)
        all_true_b.extend(labels.numpy())

macro_f1_b = f1_score(all_true_b, all_preds_b, average='macro')
print(f"Model B Macro-F1: {macro_f1_b:.4f}\n")
print(classification_report(all_true_b, all_preds_b, target_names=[str(i) for i in range(1, 11)]))

Model B Macro-F1: 0.5777

              precision    recall  f1-score   support

           1       0.76      0.79      0.78      3243
           2       0.65      0.55      0.59      1040
           3       0.51      0.60      0.55       977
           4       0.70      0.57      0.63       752
           5       0.49      0.58      0.53      1202
           6       0.73      0.49      0.58       951
           7       0.51      0.53      0.52      1418
           8       0.38      0.61      0.47      2834
           9       0.45      0.46      0.45      4130
          10       0.77      0.60      0.68      7648

    accuracy                           0.59     24195
   macro avg       0.60      0.58      0.58     24195
weighted avg       0.62      0.59      0.60     24195



In [ ]:

all_preds_b_arr = np.array(all_preds_b)
all_true_b_arr = np.array(all_true_b)
within_one_b = np.abs(all_preds_b_arr - all_true_b_arr) <= 1
print(f"Model B — Predictions within ±1 of true rating: {within_one_b.mean()*100:.1f}%")

Model B — Predictions within ±1 of true rating: 82.4%


### Model A vs Model B — Comparison

In [ ]:
#run this if you used the already-trained BERT Model
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for input_ids, attention_mask, batch_labels in val_loader:
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(batch_labels.numpy())

print(f"Model A macro-F1 (sanity check): {f1_score(all_true, all_preds, average='macro'):.4f}")  # should be 0.5511

Model A macro-F1 (sanity check): 0.5511


In [ ]:

print("Same length:", len(all_true) == len(all_true_b))
print("Same true labels, same order:", np.array_equal(np.array(all_true), np.array(all_true_b)))

Same length: True
Same true labels, same order: True


In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

all_true_arr = np.array(all_true)
preds_a = np.array(all_preds)
preds_b = np.array(all_preds_b)

correct_a = (preds_a == all_true_arr)
correct_b = (preds_b == all_true_arr)

both_correct = np.sum(correct_a & correct_b)
a_only = np.sum(correct_a & ~correct_b)
b_only = np.sum(~correct_a & correct_b)
both_wrong = np.sum(~correct_a & ~correct_b)

print(f"Both correct: {both_correct}, A only: {a_only}, B only: {b_only}, Both wrong: {both_wrong}")

table = [[both_correct, a_only], [b_only, both_wrong]]
result = mcnemar(table, exact=False, correction=True)
print(f"McNemar's statistic: {result.statistic:.4f}, p-value: {result.pvalue:.6f}")

Both correct: 12951, A only: 987, B only: 1295, Both wrong: 8962
McNemar's statistic: 41.3011, p-value: 0.000000


In [ ]:
f1_per_class_a = f1_score(all_true_arr, preds_a, average=None)
f1_per_class_b = f1_score(all_true_arr, preds_b, average=None)

improved = np.sum(f1_per_class_b > f1_per_class_a)
print(f"Classes where B > A: {improved}/10")
print(f"Per-class F1 — A: {np.round(f1_per_class_a, 3)}")
print(f"Per-class F1 — B: {np.round(f1_per_class_b, 3)}")

from scipy.stats import binomtest
result = binomtest(improved, n=10, p=0.5, alternative='greater')
print(f"Sign test p-value: {result.pvalue:.4f}")

Classes where B > A: 9/10
Per-class F1 — A: [0.762 0.536 0.506 0.58  0.506 0.545 0.483 0.445 0.483 0.665]
Per-class F1 — B: [0.776 0.593 0.551 0.629 0.529 0.584 0.519 0.467 0.454 0.675]
Sign test p-value: 0.0107


In [ ]:
from scipy.stats import wilcoxon

stat, p_value = wilcoxon(f1_per_class_b, f1_per_class_a, alternative='greater')
print(f"Wilcoxon statistic: {stat:.4f}, p-value: {p_value:.4f}")

Wilcoxon statistic: 50.0000, p-value: 0.0098


### Statistical Significance — Model A vs Model B

We tested whether Model B's improvement over Model A is genuine using three complementary tests, ranging from prediction-level to class-level evidence:

**McNemar's test** (chi-squared, on individual predictions): Model B corrected 1,295 of Model A's mistakes, while Model A corrected only 987 of Model B's — a statistically significant asymmetry (χ² = 41.30, p < 0.000001).

**Sign test** (per-class F1, 10 classes): Model B's F1 score was higher in 9 of the 10 rating classes — the only exception being class 9, where it dipped slightly (0.483 → 0.454). This is statistically significant (p = 0.0107), despite the test's small sample size (10 classes) making it a deliberately conservative check.

**Wilcoxon signed-rank test** (per-class F1, non-parametric): confirms the same result without assuming the F1 scores are normally distributed (W = 50.0, p = 0.0098).

All three tests — one based on ~24,000 individual predictions, two based on only the 10 per-class F1 scores — agree that Model B's improvement is statistically robust rather than attributable to chance. The consistency across tests with very different sample sizes and assumptions strengthens confidence in the result: this isn't an artifact of one particular metric or one particular way of slicing the data.

### Final Model: Model B (Text + drugName + condition)

Based on the statistical comparison above, **Model B is the selected model** for the 10-class task — its improvement over Model A is confirmed significant across three independent tests (McNemar's, sign test, Wilcoxon), not attributable to chance.

**Overall performance (validation set, n=24,195):**
- Accuracy: **0.59**
- Macro-F1: **0.5777**
- Weighted-F1: **0.60**
- Predictions within ±1 of true rating: **82.4%**

**Per-class F1 breakdown:**

| Rating | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| F1 | 0.78 | 0.59 | 0.55 | 0.63 | 0.53 | 0.58 | 0.52 | 0.47 | 0.45 | 0.68 |
| Support | 3243 | 1040 | 977 | 752 | 1202 | 951 | 1418 | 2834 | 4130 | 7648 |

**Key behavioral pattern**: the model performs strongest on the extreme ratings (1 and 10, F1 of 0.78 and 0.68) — the largest, most distinctly-worded classes — and weakest on the adjacent classes (8 and 9, F1 of 0.47 and 0.45). The confusion matrix shows this isn't random confusion: errors cluster among neighboring rating values (e.g., true 9s are most often predicted as 8, 9, or 10), indicating the model reliably captures overall sentiment direction but has more difficulty distinguishing fine-grained intensity among similar ratings — consistent with the class imbalance identified during EDA, where these middle-to-upper classes had comparatively less training data than the extremes.

**Architecture**: a fine-tuned DistilBERT (9 epochs, early-stopped) used as a frozen feature extractor, combined with learned embeddings for `drugName` (32-dim) and `condition` (16-dim), feeding a small classification head trained on top.

**Note on training duration**: at the point Model A's fine-tuning was stopped (9 epochs), validation macro-F1 was still improving each epoch (0.5067 → 0.5223 → 0.5511) with no clear sign of plateauing. Training was stopped here due to time constraints rather than having reached convergence — meaning both Model A and, by extension, Model B (which relies on Model A's text features) may have further headroom for improvement with additional fine-tuning epochs. This is noted as a limitation of the current results rather than a final ceiling on performance.

In [ ]:
# @title


# === Step 1: ensure the base data/split exists, since several regeneration paths depend on it ===
if 'train_df' not in dir():
    try:
        train_df = pd.read_csv('data/train_clean.csv')
    except FileNotFoundError:
        train_df = pd.read_csv('train_clean.csv')

if 'train_split' not in dir() or 'val_split' not in dir():
    train_split, val_split = train_test_split(
        train_df, test_size=0.15, stratify=train_df['rating'], random_state=42
    )
    print("Rebuilt train_split/val_split.")

MAX_LEN = 160

# === Step 2: Model A checkpoint — check the actual folder/files exist, not just a variable ===
required_a_files = ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
model_a_ok = os.path.isdir('distilbert_10class_best') and all(
    os.path.exists(f'distilbert_10class_best/{f}') for f in required_a_files
)
print(f"Model A checkpoint present: {model_a_ok}")
if not model_a_ok:
    print("  -> MISSING. Model A checkpoint must be re-uploaded — cannot be regenerated without retraining.")

# === Step 3: Model B's vocab file — regenerate from train_split if missing ===
if not os.path.exists('model_b_vocab.json'):
    print("model_b_vocab.json missing — regenerating from train_split")
    drug_to_id = {drug: i for i, drug in enumerate(train_split['drugName'].unique())}
    condition_to_id = {cond: i for i, cond in enumerate(train_split['condition'].unique())}
    with open('model_b_vocab.json', 'w') as f:
        json.dump({
            'drug_to_id': drug_to_id,
            'condition_to_id': condition_to_id,
            'num_drugs': len(drug_to_id),
            'num_conditions': len(condition_to_id)
        }, f)
    print("  -> Regenerated.")
else:
    print("model_b_vocab.json present.")
    with open('model_b_vocab.json') as f:
        vocab_data = json.load(f)
    drug_to_id, condition_to_id = vocab_data['drug_to_id'], vocab_data['condition_to_id']

# === Step 4: Model B weights — cannot be regenerated without retraining, just check ===
model_b_ok = os.path.exists('model_b_best.pt')
print(f"model_b_best.pt present: {model_b_ok}")
if not model_b_ok:
    print("  -> MISSING. Model B weights must be re-uploaded or Model B retrained.")

# === Step 5: frozen text features — regenerate via DistilBERT forward pass if missing ===
features_missing = not (os.path.exists('train_text_features.pt') and os.path.exists('val_text_features.pt'))
if features_missing and model_a_ok:
    print("Text features missing — regenerating via frozen DistilBERT (forward pass only, no training)")
    from transformers import DistilBertModel, DistilBertTokenizerFast
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    text_encoder = DistilBertModel.from_pretrained('distilbert_10class_best').to(device)
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert_10class_best')
    text_encoder.eval()
    for p in text_encoder.parameters():
        p.requires_grad = False

    train_encodings = tokenizer(train_split['review'].tolist(), truncation=True, padding=True, max_length=MAX_LEN, return_tensors='pt')
    val_encodings = tokenizer(val_split['review'].tolist(), truncation=True, padding=True, max_length=MAX_LEN, return_tensors='pt')

    @torch.no_grad()
    def extract_text_features(encodings, batch_size=32):
        feats = []
        for i in range(0, len(encodings['input_ids']), batch_size):
            ids = encodings['input_ids'][i:i+batch_size].to(device)
            mask = encodings['attention_mask'][i:i+batch_size].to(device)
            out = text_encoder(input_ids=ids, attention_mask=mask)
            feats.append(out.last_hidden_state[:, 0, :].cpu())
        return torch.cat(feats, dim=0)

    train_text_features = extract_text_features(train_encodings)
    val_text_features = extract_text_features(val_encodings)
    torch.save(train_text_features, 'train_text_features.pt')
    torch.save(val_text_features, 'val_text_features.pt')
    print("  -> Regenerated and saved.")
elif features_missing and not model_a_ok:
    print("Text features missing AND Model A unavailable — cannot regenerate. Re-upload distilbert_10class_best first.")
else:
    print("train_text_features.pt / val_text_features.pt present.")

print("\n--- Readiness check complete ---")

Model A checkpoint present: True
model_b_vocab.json present.
model_b_best.pt present: True
Text features missing — regenerating via frozen DistilBERT (forward pass only, no training)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert_10class_best
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.bias   | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  -> Regenerated and saved.

--- Readiness check complete ---


In [ ]:
import shutil
from google.colab import files

# Model A — fine-tuned DistilBERT (re-zip to be safe, in case this is a later/different session)
shutil.make_archive('distilbert_10class_best', 'zip', 'distilbert_10class_best')
files.download('distilbert_10class_best.zip')

# Model B — small head weights + drug/condition vocabulary mappings
files.download('model_b_best.pt')
files.download('model_b_vocab.json')

# Precomputed frozen DistilBERT features for train/val splits — saves re-running the expensive
# feature-extraction step in multiclass_3, since the underlying review text/split is identical;
# only the rating labels differ for the 3-class task, which can be derived fresh from train_split/val_split
files.download('train_text_features.pt')
files.download('val_text_features.pt')

print("All files downloaded — needed for multiclass_3.ipynb startup:")
print("1. distilbert_10class_best.zip  (Model A weights + tokenizer)")
print("2. model_b_best.pt + model_b_vocab.json  (Model B head + vocab)")
print("3. train_text_features.pt + val_text_features.pt  (precomputed features — skip re-extraction)")
print("4. train_clean.csv + test_clean.csv  (already have these locally)")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All files downloaded — needed for multiclass_3.ipynb startup:
1. distilbert_10class_best.zip  (Model A weights + tokenizer)
2. model_b_best.pt + model_b_vocab.json  (Model B head + vocab)
3. train_text_features.pt + val_text_features.pt  (precomputed features — skip re-extraction)
4. train_clean.csv + test_clean.csv  (already have these locally)
